# Representative PFAS exposure profiles (NHANES 2021–2023)

This notebook rebuilds the PFAS analysis of the paper's application section with the CoRP package (`../helper_functions.py`, `../bridge_sampler.py`).
The heavy steps (bridge sampling, fixed-$K$ chains, PSU bootstrap, imputations, archetype fits) run only when `RECOMPUTE = True`; otherwise their cached results are loaded from `results/`. The cached results were produced by this notebook with `RECOMPUTE = True` on 2026-11-13.
Figures are written to `figures/`.

## Notation

**Data.** Participants $i=1,\dots,n$ ($n=2551$, aged 12+) have serum concentrations $x_i\in\mathbb R_{>0}^5$ (ng/mL) of PFHxS, PFNA, n-PFOA, n-PFOS and Sm-PFOS, a PFAS subsample weight $a_i$ (`WTSPF2YR`), a stratum $h_i$ (`SDMVSTRA`, 15 strata) and a PSU $g_i$ (`SDMVPSU`, 30 PSUs in total).
Non-detects (LOD $=0.1$ ng/mL) are filled with $0.07$ in the release.

**Transform and weighted empirical measure.** With $q_i=a_i/\sum_\ell a_\ell$ and $\hat\mu=\sum_i q_i\log x_i$,
$$y_i=\log x_i-\hat\mu,\qquad \widehat P_{\rm svy}=\sum_{i=1}^n q_i\,\delta_{y_i}.$$
Distances are therefore in log (fold-change) units; a representative $z_k$ is plotted on the concentration scale as $\exp(z_k+\hat\mu)$, which is a geometric-type summary, not an arithmetic mean concentration.

**Representative measure and WAC posterior.** $Q_\theta=\sum_{k=1}^K w_k\delta_{z_k}$ with $\theta=(K,z,w)$, and
$$\Pi(K,z,w\mid x)\propto \Pi_n(K)\,\prod_{k=1}^K N_5(z_k;0,s^2I)\,\mathrm{Dir}(w;1,\dots,1)\,\exp\{-n\lambda_0\,W_2^2(\widehat P_{\rm svy},Q_\theta)\},$$
with $\lambda_0=1/2$, $s=2\times$ (average coordinate sd of $y$) and $n=2551$ (the number of participants, not an effective sample size).

**Count prior and resolution.** $\Pi_n(K)\propto\exp(-n\lambda_0 cK)$. Let $\widehat L_K=\min_{z,w}W_2^2(\widehat P_{\rm svy},Q_\theta)$ over $K$ atoms (the weighted $k$-means distortion). The penalized criterion $\widehat L_K+cK$ is minimized at $K^*(c)$, and
$$c_K=\tfrac12(\widehat L_{K-1}-\widehat L_{K+1})$$
lies between the two adjacent drops of $\widehat L_K$, so $c=c_K$ gives $K^*=K$. The price $c$ is a resolution choice: the data form one continuum, so no single $K$ is intrinsic. The values $c_K$ are computed from the same data, so the concentration of $\Pi(K\mid x)$ on $K$ at $c=c_K$ holds by construction and is not evidence for that $K$. What the data determine is the map from $c$ to the count (Figure 2(b)) and, given a count, the profiles, the weights and their uncertainty.

**Count posterior.** $\Pi(K\mid x)\propto \Pi_n(K)Z_K$ with $Z_K=\int \Pi(z,w\mid K)\exp\{-n\lambda_0W_2^2\}\,dz\,dw$, estimated by bridge sampling from fixed-$K$ chains (`bridge_sampler.py`). $Z_K$ does not involve $\Pi_n(K)$, so one set of runs gives $\Pi(K\mid x)$ for every $c$.

**Design-based sandwich.** Conditional on $K$, with $\vartheta=(z,w_1,\dots,w_{K-1})$ and posterior draws relabelled to the best Lloyd quantizer, `calibrate(..., design=(strata, psu))` replaces the posterior covariance $S$ by $\widehat\Omega/n$, where $\widehat\Omega$ is $n$ times the with-replacement linearization variance of the influence totals over PSUs within strata: $\tilde\vartheta=\bar\vartheta+A(\vartheta-\bar\vartheta)$ with $ASA^\top=\widehat\Omega/n$.
Intervals use the $t_{15}$ quantile (30 PSUs $-$ 15 strata degrees of freedom), i.e. the calibrated draws are widened by $t_{15,0.975}/z_{0.975}$.
The paper's asymptotic normality theorem is for iid sampling; the survey version is an extension checked here against a Rao–Wu PSU bootstrap (200 replicates) of weighted $k$-means.

In [1]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from multiprocessing import get_context
from scipy.special import logsumexp
from scipy.stats import t as t_dist, norm, truncnorm
from scipy.spatial.distance import cdist

HERE = os.path.abspath('')
sys.path[:0] = [HERE, os.path.dirname(HERE)]
from helper_xpt import read_xpt
from helper_functions import Model, Exp, GaussianAtoms, Dirichlet, penalty, quantizer, lloyd_start, relabel, run, calibrate
from bridge_sampler import evidence

RECOMPUTE = False                      # True: rerun the heavy steps (about 5 minutes on 8 cores) and overwrite results/
RES, FIG = os.path.join(HERE, 'results'), os.path.join(os.path.dirname(HERE), 'figures')   # figures/ of CoRP: the paper's figures
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11, 'xtick.labelsize': 10,
                     'ytick.labelsize': 10, 'legend.fontsize': 9, 'figure.dpi': 100, 'savefig.dpi': 200,
                     'savefig.bbox': 'tight'})
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True)
save = lambda fig, name: fig.savefig(os.path.join(FIG, name + '.pdf'), bbox_inches='tight', pad_inches=0.02)   # paper figures only
PAPER = {'font.size': 7.5, 'axes.titlesize': 8, 'axes.labelsize': 7.5, 'legend.fontsize': 6.5, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
         'axes.linewidth': 0.6, 'lines.linewidth': 1.0, 'figure.dpi': 200, 'pdf.fonttype': 42}      # paper figures: text width W inches
PW = 6.3
LINE = ['#D77289', '#73BF9F', '#68A5D2', '#E4BD58', '#9B82C5', '#E59160', '#67BBBF', '#AAC662']   # same hues, deeper (lines)
import colorsys
soft = lambda C: [colorsys.hls_to_rgb(h, l, .65 * s) for h, l, s in (colorsys.rgb_to_hls(*c[:3]) for c in C)]   # viridis at 0.65 saturation
cols = lambda K: soft(plt.cm.viridis(np.linspace(.1, .9, K)))   # profile colours: lowest burden dark, highest light
FILL = soft(plt.cm.viridis(np.linspace(0, 1, 8)))   # count colours K = 1..8 (areas), same family as the profiles

## 1. Analytic sample

Join `PFAS_L` and `DEMO_L` by `SEQN`; keep participants with a positive PFAS weight and all five results.

In [2]:
VARS = ['LBXPFHS', 'LBXPFNA', 'LBXNFOA', 'LBXNFOS', 'LBXMFOS']
FLAGS = ['LBDPFHSL', 'LBDPFNAL', 'LBDNFOAL', 'LBDNFOSL', 'LBDMFOSL']   # 1 = below LOD
NAMES = ['PFHxS', 'PFNA', 'n-PFOA', 'n-PFOS', 'Sm-PFOS']
pf, de = read_xpt(os.path.join(HERE, 'PFAS_L.xpt')), read_xpt(os.path.join(HERE, 'DEMO_L.xpt'))
row = {s: i for i, s in enumerate(de['SEQN'])}
r = np.array([row[s] for s in pf['SEQN']])
X, F, a = np.c_[[pf[v] for v in VARS]].T, np.c_[[pf[f] for f in FLAGS]].T, pf['WTSPF2YR']
keep = (a > 0) & ~np.isnan(X).any(1)
X, F, a = X[keep], F[keep], a[keep]
age, stra, psu, race, educ, pir = (de[v][r[keep]] for v in ('RIDAGEYR', 'SDMVSTRA', 'SDMVPSU', 'RIDRETH3', 'DMDEDUC2', 'INDFMPIR'))
n, q = len(a), a / a.sum()
print(f'PFAS subsample rows {len(pf["SEQN"])}; positive weight {(pf["WTSPF2YR"] > 0).sum()}; analytic n {n}')
print(f'strata {len(np.unique(stra))}, PSUs {len(np.unique(stra * 10 + psu))}, Kish effective n {1 / (q ** 2).sum():.0f}, '
      f'all five detected {np.mean(~(F == 1).any(1)):.3f}, age {age.min():.0f}-{age.max():.0f}')
C0 = np.load(os.path.join(RES, 'analytic.npz'))
print('identical to the cached analytic sample:', all(np.array_equal(C0[k], v) for k, v in
                                                      dict(X=X, F=F, a=a, age=age, stra=stra, psu=psu).items()))

PFAS subsample rows 3618; positive weight 2551; analytic n 2551
strata 15, PSUs 30, Kish effective n 1679, all five detected 0.936, age 12-80
identical to the cached analytic sample: True


In [3]:
LAM0 = 0.5
logX = np.log(X)
mu = q @ logX
Y = logX - mu                          # fit coordinates: centred log concentrations


def model(c=0.0):
    return Model(Y, 2, Exp(n * LAM0), penalty(n * LAM0 * c), GaussianAtoms(np.zeros(5), 2 * Y.std(0).mean()),
                 Dirichlet(1.0), K_max=16, weights=a)


ev, U = np.linalg.eigh(np.cov(Y.T, aweights=q))
ev, U = ev[::-1] / ev.sum(), U[:, ::-1][:, :2] * np.sign(U[:, ::-1][:, :2].sum(0))
print('share of variance of the first two weighted principal components', ev[:2].round(3), '; PC1 loadings', U[:, 0].round(2))

share of variance of the first two weighted principal components [0.784 0.083] ; PC1 loadings [0.49 0.34 0.36 0.49 0.53]


## 2. Resolution: weighted distortion $\widehat L_K$ and the price $c$

In [4]:
if RECOMPUTE:
    M = model()
    np.savez(os.path.join(RES, 'distortion.npz'), Ls=[quantizer(M, K)[0] for K in range(1, 12)])
Ls = np.load(os.path.join(RES, 'distortion.npz'))['Ls']
c_mid = {K: (Ls[K - 2] - Ls[K]) / 2 for K in range(2, 11)}
print('L_K, K = 1..11:', Ls.round(4))
print('c giving K* = 3..7:', {K: round(c_mid[K], 3) for K in range(3, 8)})

L_K, K = 1..11: [3.5878 1.7797 1.3315 1.1033 0.9845 0.9157 0.8548 0.8108 0.7681 0.7358
 0.7017]
c giving K* = 3..7: {3: np.float64(0.338), 4: np.float64(0.174), 5: np.float64(0.094), 6: np.float64(0.065), 7: np.float64(0.052)}


## 3. Count posterior by bridge sampling

Two fixed-$K$ chains (seeds 0, 1; 3000 iterations) for each $K=1,\dots,16$ (the $K_{\max}$ of the model); $\log Z_K$ is the average over the two chains. The figures report $K\le8$ one by one and the posterior mass on $K>8$ as one grey band.

In [5]:
KS_Z = np.arange(1, 17)


def ev_job(K, s):
    e = evidence(model(), K, 3000, seed=s)
    return K, s, e['logZ'], e['re']


if RECOMPUTE:
    with get_context('fork').Pool(8) as pool:
        np.savez(os.path.join(RES, 'kmix.npz'), out=np.array(pool.starmap(ev_job, [(K, s) for K in KS_Z for s in (0, 1)])))
O = np.load(os.path.join(RES, 'kmix.npz'))['out']
logZ = np.array([O[O[:, 0] == K, 2].mean() for K in KS_Z])
print('K: (largest relative error of Z_K over the two chains, |log Z_K(seed 0) - log Z_K(seed 1)|)')
print({int(K): (round(float(O[O[:, 0] == K, 3].max()), 3), round(float(np.ptp(O[O[:, 0] == K, 2])), 2)) for K in KS_Z})


def log_post_K(c):
    l = logZ - n * LAM0 * c * KS_Z
    return l - logsumexp(l)

K: (largest relative error of Z_K over the two chains, |log Z_K(seed 0) - log Z_K(seed 1)|)
{1: (0.004, 0.0), 2: (0.008, 0.0), 3: (0.018, 0.05), 4: (0.024, 0.02), 5: (0.044, 0.06), 6: (0.074, 0.18), 7: (0.074, 0.15), 8: (0.115, 0.03), 9: (0.485, 1.9), 10: (0.219, 0.12), 11: (0.283, 0.21), 12: (1.193, 1.13), 13: (0.603, 2.35), 14: (0.818, 2.22), 15: (1.255, 1.89), 16: (1.175, 6.12)}


## 4. Fixed-$K$ chains, design sandwich and PSU bootstrap ($K=3,\dots,7$)

* **chains**: two fixed-$K$ chains per $K$ (no count moves), each calibrated with the design sandwich; profiles are ordered from lowest to highest overall burden.
* **boot**: Rao–Wu PSU bootstrap ($n_h-1$ PSUs drawn per stratum) of the weighted quantizer, aligned to the chain posterior mean; per replicate the atoms, Voronoi weights and age-group shares.

In [6]:
KS, B, LOD = range(3, 8), 200, 0.1
AGES, AGE_LAB = [(12, 20), (20, 40), (40, 60), (60, 200)], ['12–19', '20–39', '40–59', '60+']


def chain_job(K, s):
    M = model()
    r = run(M, 3000, lloyd_start(M, K, s), split_merge=False, temper=False, burn=1 / 3, thin=2, seed=s)
    cd = calibrate(M, r['draws'], K, design=(stra, psu))
    o = np.argsort(cd['z_bar'].sum(1))
    return {f'K{K}_s{s}_zbar': cd['z_bar'][o], f'K{K}_s{s}_wbar': cd['w_bar'][o],
            f'K{K}_s{s}_draws': cd['z'][:, o], f'K{K}_s{s}_wdraws': cd['w'][:, o]}


def shares(x, wt, z):
    lab = cdist(x, z, 'sqeuclidean').argmin(1)
    g = [(age >= l) & (age < u) for l, u in AGES]
    return (np.bincount(lab, wt, minlength=len(z)) / wt.sum(),
            np.array([np.bincount(lab[s], wt[s], minlength=len(z)) / wt[s].sum() for s in g]))


def fit_aligned(x, wt, ref, seed):
    out = {}
    for K, zr in ref.items():
        z = quantizer(Model(x, 2, Exp(1.0), weights=wt), K, n_init=5, seed=seed)[1]
        z = z[relabel(zr, z)]
        out[K] = (z, *shares(x, wt, z))
    return out


def rw_mass(b):
    """Rao-Wu multipliers of PSU bootstrap replicate b: n_h - 1 PSUs drawn per stratum, weight x n_h / (n_h - 1) x times drawn."""
    rng, m = np.random.default_rng(b), np.zeros(n)
    for h in np.unique(stra):
        P = np.unique(psu[stra == h])
        cnt = np.bincount(rng.integers(0, len(P), len(P) - 1), minlength=len(P))
        for k, p in enumerate(P):
            m[(stra == h) & (psu == p)] = cnt[k] * len(P) / (len(P) - 1)
    return m


def boot_job(b, ref):
    return fit_aligned(Y, a * rw_mass(b), ref, b)


if RECOMPUTE:
    with get_context('fork').Pool(8) as pool:
        res = pool.starmap(chain_job, [(K, s) for K in KS for s in (0, 1)])
        np.savez(os.path.join(RES, 'byK_chains.npz'), **{k: v for d in res for k, v in d.items()})
        ref = {K: np.load(os.path.join(RES, 'byK_chains.npz'))[f'K{K}_s0_zbar'] for K in KS}
        res = pool.starmap(boot_job, [(b, ref) for b in range(B)])
        np.savez(os.path.join(RES, 'byK_boot.npz'), **{f'K{K}_{v}': np.array([r[K][i] for r in res])
                                                      for K in KS for i, v in enumerate(('z', 'w', 'age'))})
C, Bt = (np.load(os.path.join(RES, f'byK_{m}.npz')) for m in ('chains', 'boot'))
TF = t_dist.ppf(.975, 15) / norm.ppf(.975)
pct = lambda v: np.percentile(v, [2.5, 97.5], 0)


def get(K):
    """posterior means (log ng/mL) and t_15-widened design-calibrated draws of z_k (log ng/mL) and w_k, seed-0 chain."""
    zb, wb = C[f'K{K}_s0_zbar'], C[f'K{K}_s0_wbar']
    return zb + mu, wb, zb + mu + TF * (C[f'K{K}_s0_draws'] - zb), wb + TF * (C[f'K{K}_s0_wdraws'] - wb)


for K in KS:
    zb, wb, Z, W = get(K)
    cross = sum(np.any(np.diff(np.sign(zb[i] - zb[j])) != 0) for i in range(K) for j in range(i + 1, K))
    print(f'K {K}: w {wb.round(3).tolist()}; crossing profile pairs {cross}; '
          f'max |zbar(seed 0) - zbar(seed 1)| {np.abs(C[f"K{K}_s0_zbar"] - C[f"K{K}_s1_zbar"]).max():.3f}\n'
          f'   sd of w: sandwich {(W.std(0) / TF).round(3).tolist()}, bootstrap {Bt[f"K{K}_w"].std(0).round(3).tolist()}')

K 3: w [0.219, 0.433, 0.348]; crossing profile pairs 0; max |zbar(seed 0) - zbar(seed 1)| 0.008
   sd of w: sandwich [0.054, 0.014, 0.048], bootstrap [0.037, 0.017, 0.045]
K 4: w [0.107, 0.313, 0.372, 0.208]; crossing profile pairs 0; max |zbar(seed 0) - zbar(seed 1)| 0.013
   sd of w: sandwich [0.034, 0.011, 0.024, 0.016], bootstrap [0.025, 0.014, 0.022, 0.016]
K 5: w [0.047, 0.198, 0.271, 0.308, 0.177]; crossing profile pairs 0; max |zbar(seed 0) - zbar(seed 1)| 0.007
   sd of w: sandwich [0.01, 0.022, 0.013, 0.021, 0.016], bootstrap [0.011, 0.022, 0.014, 0.019, 0.017]
K 6: w [0.044, 0.168, 0.209, 0.188, 0.245, 0.147]; crossing profile pairs 1; max |zbar(seed 0) - zbar(seed 1)| 0.029
   sd of w: sandwich [0.01, 0.03, 0.018, 0.037, 0.015, 0.029], bootstrap [0.01, 0.026, 0.035, 0.042, 0.041, 0.041]
K 7: w [0.041, 0.136, 0.21, 0.159, 0.159, 0.183, 0.112]; crossing profile pairs 3; max |zbar(seed 0) - zbar(seed 1)| 0.034
   sd of w: sandwich [0.009, 0.008, 0.017, 0.028, 0.027, 0.031, 0.0

## Figure 1. Pairwise view of the data with the $K=4$ representatives

In [7]:
zb4, wb4, _, _ = get(4)
PCOL = plt.cm.viridis(np.linspace(.05, .9, 4))   # full-saturation viridis: profile 1 stays visible on the gray points
L = logX
wcor = lambda u, v: np.cov(u, v, aweights=q)[0, 1] / np.sqrt(np.cov(u, aweights=q) * np.cov(v, aweights=q))
with plt.rc_context(PAPER):
    fig, ax = plt.subplots(5, 5, figsize=(PW, PW * 0.98))
    for i in range(5):
        for j in range(5):
            A = ax[i, j]
            if i == j:
                A.hist(L[:, i], 40, weights=q, density=True, color='0.75')
                A.hist(L[:, i], 40, density=True, histtype='step', color='k', lw=0.8)
                A.set_yticks([])
            elif i > j:
                A.scatter(L[:, j], L[:, i], s=2.5, c='0.7', alpha=.3, lw=0, rasterized=True)             # all participants
                A.scatter(zb4[:, j], zb4[:, i], s=45 + 160 * wb4, c=PCOL, edgecolor='k', lw=.6, alpha=.5, zorder=4)
            else:
                A.text(.5, .5, f'{wcor(L[:, j], L[:, i]):.2f}\n({np.corrcoef(L[:, j], L[:, i])[0, 1]:.2f})', ha='center', va='center', fontsize=8, transform=A.transAxes)
                A.set_xticks([]); A.set_yticks([])
            A.tick_params(labelleft=j == 0 and i > 0, labelbottom=i == 4)   # tick labels only on the outer panels
            if i == 4: A.set_xlabel(f'log {NAMES[j]}')
            if j == 0 and i > 0: A.set_ylabel(f'log {NAMES[i]}')
    ax[0, 0].set_ylabel('density')
    handles = ([Line2D([], [], marker='s', ls='', ms=7, color='0.75', label='survey-weighted histogram'), Line2D([], [], color='k', lw=0.6, label='unweighted histogram'),
                Line2D([], [], marker='o', ls='', ms=3, color='0.55', alpha=.6, label='participant')] +
               [Line2D([], [], marker='o', ls='', ms=np.sqrt(45 + 160 * w), mfc=c, mec='k', alpha=.5, label=f'profile {k + 1}, $\\bar w$ = {w:.2f}') for k, (c, w) in enumerate(zip(PCOL, wb4))])
    fig.legend(handles=handles, loc='lower center', ncol=4, bbox_to_anchor=(.5, -0.06), frameon=False, labelspacing=1.4, borderpad=0.2)
    fig.subplots_adjust(bottom=.1, wspace=0.1, hspace=0.1)
    save(fig, 'fig_pfas_pairs')

## Figure 2. How the price $c$ selects the resolution $K^*$, and the count posterior along $c$

(a) A line of slope $-c$ supports the distortion curve at $K^*(c)=\arg\min_K \widehat L_K+cK$. (b) The count posterior $\Pi(K\mid x)\propto\Pi_n(K)Z_K$ for each $c$, from one set of bridge-sampling runs ($\log Z_K$ for $K=1,\dots,16$, reported for $K\le8$, with the mass on $K>8$ as one grey band). Each vertical slice is the count posterior under the prior with that $c$. The triangles mark the switch points, where the posterior mode moves from a larger count $a$ to a smaller count $b$ and the two posteriors tie, $c=(\log Z_a-\log Z_b)/\{n\lambda_0(a-b)\}$. Colors of $K$ are shared by both panels and Figure 2b.

Reliability of the evidence. The two chains agree to within 0.2 in $\log Z_K$ for $K\le8$ and $K=10$, but not for $K=9$ (1.90), $K=11$ (0.21) or $K=12,\dots,16$ (1.1 to 6.1), where the relative errors of $Z_K$ are 0.2 to 1.3: the fixed-$K$ chains mix poorly with many atoms at $n=2551$ in $d=5$. These counts enter the figures only through the grey $K>8$ band, which holds all the mass for $c\le0.03$ and almost none for $c\ge0.035$, so only the left edge of the figure is uncertain.

In [8]:
import matplotlib.patheffects as pe
cg = np.geomspace(.02, 2.5, 800)
Pk = np.exp([log_post_K(c) for c in cg]).T
nl = n * LAM0
mode = KS_Z[Pk.argmax(0)]
Lz = dict(zip(KS_Z, logZ))
crit = [((Lz[a] - Lz[b]) / (nl * (a - b)), a, b) for a, b in zip(mode[:-1], mode[1:]) if a != b and a <= 8]   # (c where Pi(K=a|x) = Pi(K=b|x), a, b), reported for a <= 8
kc = dict(zip(range(1, 9), FILL))                                        # band colours for K <= 8, grey above
P8 = np.r_[Pk[:8], Pk[8:].sum(0, keepdims=True)]
with plt.rc_context(PAPER):
    fig, ax = plt.subplots(1, 2, figsize=(PW, 2.5), gridspec_kw=dict(width_ratios=[1, 1.25]))
    A, Ks = ax[0], np.arange(1, len(Ls) + 1)
    A.plot(Ks, Ls, 'ko-', ms=2.5, label=r'$\widehat L_K$')
    for K in KS:
        kk = np.array([K - 2.2, K + 2.2])
        A.plot(kk, Ls[K - 1] - c_mid[K] * (kk - K), color=LINE[K - 1], lw=1.2, label=f'$c_{K}={c_mid[K]:.3f}$')
        A.plot(K, Ls[K - 1], 'o', color=LINE[K - 1], ms=5)
    A.set(xlabel='$K$', ylabel=r'$\widehat L_K$', xticks=Ks, ylim=(.55, 2.1), xlim=(.5, 11.5))
    A.legend()
    A = ax[1]
    A.stackplot(cg, P8, colors=[*FILL, '0.75'], edgecolor='w', lw=0.4)
    for k in range(9):
        m = P8[k] > 0.5
        if m.any() and cg[m][-1] / cg[m][0] > 1.5:
            A.text(np.sqrt(cg[m][0] * cg[m][-1]), 0.5, f'$K={k + 1}$' if k < 8 else '$K>8$', ha='center', va='center', fontsize=6.5)
    for cc, a, b in crit:
        A.plot(cc, 0, '^', ms=5, color=kc[a], mec='k', mew=0.5, clip_on=False, zorder=6)
    C_SEL = 0.150                                                             # chosen price: log-midpoint of the K = 4 interval, sqrt(0.105 x 0.214)
    A.axvline(C_SEL, color='#E8143C', ls='--', lw=0.9)
    A.text(C_SEL * 1.04, .97, f'$c={C_SEL:.3f}$', va='top', fontsize=6.5, color='#E8143C', path_effects=[pe.withStroke(linewidth=2, foreground='w')])
    A.set(xscale='log', xlim=(cg[0], cg[-1]), ylim=(0, 1), xlabel='$c$', ylabel=r'$\Pi(K\mid x_{1:n})$')
    A.set_xticks([.02, .05, .1, .2, .5, 1, 2], ['0.02', '0.05', '0.1', '0.2', '0.5', '1', '2'])
    A.set_xticks([], minor=True)
    A.plot([], [], '^', ms=5, c='0.6', mec='k', label='switch point')
    A.legend(loc='center left', bbox_to_anchor=(1.0, .5), handlelength=1.2, borderpad=0.3)
    fig.tight_layout(w_pad=0.8)
    save(fig, 'fig_pfas_c')
for cc, a, b in crit:
    print(f'switch K = {a} -> {b} at c = {cc:.4f}; prior odds Pi_n(K+1)/Pi_n(K) = 10^{-nl * cc / np.log(10):.1f}')

switch K = 8 -> 7 at c = 0.0344; prior odds Pi_n(K+1)/Pi_n(K) = 10^-19.1
switch K = 7 -> 6 at c = 0.0498; prior odds Pi_n(K+1)/Pi_n(K) = 10^-27.6
switch K = 6 -> 5 at c = 0.0579; prior odds Pi_n(K+1)/Pi_n(K) = 10^-32.1
switch K = 5 -> 4 at c = 0.1054; prior odds Pi_n(K+1)/Pi_n(K) = 10^-58.4
switch K = 4 -> 3 at c = 0.2137; prior odds Pi_n(K+1)/Pi_n(K) = 10^-118.4
switch K = 3 -> 2 at c = 0.4337; prior odds Pi_n(K+1)/Pi_n(K) = 10^-240.3
switch K = 2 -> 1 at c = 1.7904; prior odds Pi_n(K+1)/Pi_n(K) = 10^-991.8


## Figure 2b. Prior and posterior of $K$ at each switch point

One column per switch point (the triangles in Figure 2(b)). Row 1: the one-observation count prior $\Pi_1(K)\propto e^{-\lambda_0cK}$, a geometric distribution truncated to $K\le16$. Row 2: the count prior for the $n=2551$ observations, $\Pi_n(K)\propto\Pi_1(K)^n=e^{-n\lambda_0cK}$. Rows 3 to 5: the count posterior $\Pi(K\mid x)\propto\Pi_n(K)Z_K$, computed for $K=1,\dots,16$ (grey bars: $K>8$), just below the switch point, at it, and just above it, with $\delta=5/(n\lambda_0)=0.004$.

$\Pi_1$ is mild and close to uniform for small $c$. Its $n$-th power $\Pi_n$ is a point mass at $K=1$, since $\Pi_n(K+1)/\Pi_n(K)=e^{-n\lambda_0c}$ with $n\lambda_0=1275.5$. The evidence grows at the same order, $Z_K/Z_{K-1}\approx e^{n\lambda_0(\widehat L_{K-1}-\widehat L_K)}$, so the posterior compares the error reduction per observation with $c$. For almost every $c$ the posterior is a single spike. A switch point is by definition the one value of $c$ at which two neighboring counts tie, so only there is the posterior split 1/2 to 1/2. Moving $c$ by $\delta$ changes the log posterior odds of the two counts by 5, which already gives a spike of about 0.99 on the larger count below the switch point and on the smaller count above it. The exception is the first column: the $K=8$ band is so narrow (Figure 2(b)) that at $c-\delta$ most of the mass has already moved to $K=9$, whose evidence is unreliable (Figure 2 notes). The two seeds agree within 0.2 in $\log Z_K$ for every $K\le8$.

In [9]:
kk, dl = np.arange(1, 17), 5 / nl
fig, ax = plt.subplots(5, len(crit), figsize=(3 * len(crit), 12), sharex=True, sharey='row')
for j, (cc, a, b) in enumerate(crit):
    p1, pn = np.exp(-LAM0 * cc * kk), np.exp(-nl * cc * (kk - 1))
    for r, (K, p) in enumerate(((kk, p1 / p1.sum()), (kk, pn / pn.sum()), *((KS_Z, np.exp(log_post_K(cc + e))) for e in (-dl, 0, dl)))):
        ax[r, j].bar(K, p, color=[kc.get(k, '0.75') for k in K], ec='k', lw=.4)
    ax[0, j].set_title(f'c = {cc:.3f}:  K = {a} vs {b}', fontsize=11)
    ax[4, j].set(xticks=[1, 4, 8, 12, 16], xlabel='K')
for r, lab in enumerate((r'$\Pi_1(K)$ (one observation)', r'prior $\Pi_n(K)$, n = 2551', r'posterior at $c-\delta$', r'posterior at $c$', r'posterior at $c+\delta$')):
    ax[r, 0].set_ylabel(lab)
for r in range(1, 5):
    ax[r, 0].set_ylim(0, 1.05)
fig.suptitle(f'Count prior and posterior at each switch point, $\\delta = 5/(n\\lambda_0)$ = {dl:.4f} (bar color = K as in Figure 2; grey: K > 8)')
fig.tight_layout()

## Figure 4. Main figure: count, profiles and weights at $K=4$

In [10]:
zb, wb, Z, W = get(4)
x5, col = np.arange(5), cols(4)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8), gridspec_kw=dict(width_ratios=[1, 1.6, 1]))
A = ax[0]
A.axvspan(8.5, 9.5, color='0.9', lw=0)
for K, m in zip((3, 4, 5), 'sov'):
    lp = log_post_K(c_mid[K])
    l, = A.plot(KS_Z[:8], lp[:8] / np.log(10), marker=m, label=f'c = {c_mid[K]:.3f}  (K* = {K})')
    A.plot(9, logsumexp(lp[8:]) / np.log(10), marker=m, color=l.get_color(), mfc='none')
A.axhline(np.log10(.05), color='0.6', ls=':', lw=1, label=r'$\Pi = 0.05$')
A.set(xlabel='number of representatives K', ylabel=r'$\log_{10}\Pi(K\mid x)$', ylim=(-60, 4), xticks=range(1, 10), xticklabels=[*range(1, 9), '>8'],
      title='(a) count posterior at three prices c')
A.legend(loc='upper right')
A = ax[1]
for j in range(4):
    A.fill_between(x5, *np.exp(pct(Z[:, j])), color=col[j], alpha=.3, lw=0)
    A.plot(x5, np.exp(zb[j]), color=col[j], lw=1 + 15 * wb[j], marker='o', ms=4 + 15 * wb[j],
           label=f'profile {j + 1}: w = {wb[j]:.2f}')
A.set(yscale='log', xticks=x5, xticklabels=NAMES, ylabel='serum concentration (ng/mL, log scale)',
      title='(b) K = 4 representative profiles $z_k$\n(width ∝ $w_k$; bands: pointwise 95% design-calibrated intervals)')
A.legend(loc='upper left', ncol=2)
A.set_ylim(top=A.get_ylim()[1] * 2.5)
A = ax[2]
lo, hi = pct(W)
for j in range(4):
    A.errorbar(wb[j], j, xerr=[[wb[j] - lo[j]], [hi[j] - wb[j]]], fmt='o', color=col[j], capsize=4, ms=8)
A.set(yticks=range(4), yticklabels=['profile 1\n(lowest)', 'profile 2', 'profile 3', 'profile 4\n(highest)'],
      xlabel='population share $w_k$', xlim=(0, .5), title='(c) weights, 95% design-calibrated ($t_{15}$)')
fig.tight_layout()

## Figure 5. Representative profiles for $K=3,\dots,7$

Thick lines are the posterior means $\bar z_k$, with width proportional to $\bar w_k$. The bands are pointwise 95% intervals for each coordinate of $z_k$ from the design-calibrated posterior draws ($t_{15}$-widened), not observations.

In [11]:
x5 = np.arange(5)
with plt.rc_context(PAPER):
    fig, ax = plt.subplots(1, len(KS), figsize=(PW, 2.6), sharey=True)
    for A, K in zip(ax, KS):
        zb, wb, Z, Wd = get(K)
        col = cols(K)
        for j in range(K):
            A.fill_between(x5, *np.exp(pct(Z[:, j])), color=col[j], alpha=.3, lw=0)
            A.plot(x5, np.exp(zb[j]), color=col[j], lw=0.6 + 7 * wb[j], marker='o', ms=1.5 + 7 * wb[j], label=f'{wb[j]:.2f}')
        A.set(yscale='log', xticks=x5, title=f'$K={K}$')
        A.set_xticklabels(NAMES, rotation=50, ha='right', fontsize=6)
        A.legend(loc='upper left', title='$\\bar w_k$', title_fontsize=6, fontsize=5.5, handlelength=1.2, borderpad=0.2, labelspacing=0.15, ncol=2 if K > 5 else 1, columnspacing=0.6)
    ax[0].set_ylabel('serum concentration (ng/mL)')
    fig.tight_layout(w_pad=0.4)
    save(fig, 'fig_pfas_profiles')

## Figure 6. Weights: design sandwich versus PSU bootstrap

In [12]:
with plt.rc_context(PAPER):
    fig, ax = plt.subplots(1, len(KS), figsize=(PW, 2.2), sharex=True)
    for A, K in zip(ax, KS):
        zb, wb, Z, Wd = get(K)
        col, (lo, hi), (blo, bhi) = cols(K), pct(Wd), pct(Bt[f'K{K}_w'])
        for j in range(K):
            A.errorbar(wb[j], j + .17, xerr=[[wb[j] - lo[j]], [hi[j] - wb[j]]], fmt='o', ms=3, color=col[j], capsize=1.5, lw=0.8)
            A.errorbar(wb[j], j - .17, xerr=[[wb[j] - blo[j]], [bhi[j] - wb[j]]], fmt='s', ms=3, mfc='w', color=col[j], capsize=1.5, lw=0.8)
        A.set(yticks=range(K), yticklabels=[f'{j + 1}' for j in range(K)], xlabel='$w_k$', title=f'$K={K}$', xlim=(0, .5), ylim=(-.6, K - .4))
    ax[0].set_ylabel('profile (lowest to highest)')
    fig.legend(handles=[Line2D([], [], marker='o', ms=3, color='k', label='design sandwich, 95% ($t_{15}$)'), Line2D([], [], marker='s', ms=3, mfc='w', color='k', label='Rao–Wu PSU bootstrap of weighted $k$-means, 95%')],
               loc='lower center', ncol=2, bbox_to_anchor=(.5, -0.06), frameon=False)
    fig.tight_layout(w_pad=0.4)
    save(fig, 'fig_pfas_weights')

## Figure 7. Allocation of demographic groups to the profiles

Each participant is allocated to the nearest representative (Voronoi cell). For each demographic group, the bars show the posterior median of the survey-weighted share of the group in each profile over the design-calibrated WAC posterior draws of the representatives ($t_{15}$-widened, as in Figures 4 and 5): for every draw $z^{(s)}$ each participant is reallocated to the nearest representative, and the 95% intervals are the 2.5% and 97.5% percentiles over draws. These intervals reflect the uncertainty in the representatives only. They do not include the sampling variability of the group shares given the representatives, so they are narrower than intervals that also resample participants, such as the PSU bootstrap. The groups are the four age groups, race and Hispanic origin (`RIDRETH3`, Mexican American and other Hispanic combined), education among adults aged 20 and over (`DMDEDUC2`, asked only of adults) and the family income to poverty ratio (`INDFMPIR`, with the usual NHANES cut points 1.3 and 3.5, missing for 302 participants).

Definitions of the two survey variables:

- **Race and Hispanic origin** (`RIDRETH3`). NHANES treats Hispanic origin as an ethnicity rather than a race, so Hispanic participants (Mexican American and other Hispanic, codes 1 and 2, combined here) are separated first, and the remaining participants are classified by race as non-Hispanic (NH) White, NH Black, NH Asian (codes 3, 4, 6) and other or multiracial (code 7).
- **Family income to poverty ratio** (`INDFMPIR`). Annual family income divided by the federal poverty guideline for a family of that size and state, so that families of different sizes are comparable. A ratio of 1 is income exactly at the poverty line, values below 1 are poverty, and NHANES top-codes the ratio at 5. The cut points 1.3 (the income eligibility limit of SNAP and other assistance programs) and 3.5 give the usual low, middle and high income groups.


In [13]:
GROUPS = {
    'age': ('age group (years)', AGE_LAB, [(age >= l) & (age < u) for l, u in AGES]),
    'race': ('race and Hispanic origin', ['Hispanic', 'NH White', 'NH Black', 'NH Asian', 'other'],
             [np.isin(race, (1, 2)), race == 3, race == 4, race == 6, race == 7]),
    'educ': ('education, adults aged 20+', ['< high school', 'high school', 'some college', 'college+'],
             [np.isin(educ, (1, 2)), educ == 3, educ == 4, educ == 5]),
    'pir': ('family income to poverty ratio', ['< 1.3', '1.3–3.5', '≥ 3.5'], [pir < 1.3, (pir >= 1.3) & (pir < 3.5), pir >= 3.5]),
}
def group_shares(K, groups):
    """design-calibrated posterior draws x groups x profiles: survey-weighted share of each group allocated to each
    profile, with participants reallocated to the nearest representative of every draw z^(s)."""
    zb = C[f'K{K}_s0_zbar']
    Zs = zb + TF * (C[f'K{K}_s0_draws'] - zb)                             # t_15-widened draws, centred log scale
    out = np.empty((len(Zs), len(groups), K))
    for s, z in enumerate(Zs):
        lab = cdist(Y, z, 'sqeuclidean').argmin(1)
        out[s] = [np.bincount(lab[g], q[g], minlength=K) / q[g].sum() for g in groups]
    return out


K4 = 4
with plt.rc_context(PAPER):                                               # paper figure: the four groupings at K = 4
    fig, ax = plt.subplots(1, 4, figsize=(PW, 2.5), sharey=True)
    for A, (xlabel, labels, groups) in zip(ax, GROUPS.values()):
        Ag = group_shares(K4, groups)
        m, (lo, hi), col, wd = np.median(Ag, 0), pct(Ag), cols(K4), .8 / K4
        for j in range(K4):
            xs = np.arange(len(groups)) + (j - (K4 - 1) / 2) * wd
            A.bar(xs, m[:, j], wd, color=col[j], label=f'profile {j + 1}')
            A.errorbar(xs, m[:, j], yerr=[m[:, j] - lo[:, j], hi[:, j] - m[:, j]], fmt='none', ecolor='k', lw=.5, capsize=1)
        A.set(xticks=range(len(groups)), xlabel=xlabel.replace(' group (years)', ' (years)').replace('family ', ''), ylim=(0, 0.8))
        A.set_xticklabels(labels, rotation=30, ha='right', fontsize=5.5)
        print(xlabel, 'median shares', np.round(m, 2).tolist(), 'lo', np.round(lo, 2).tolist(), 'hi', np.round(hi, 2).tolist())
    ax[0].set_ylabel('share of group')
    fig.align_xlabels(ax)
    fig.legend(*ax[0].get_legend_handles_labels(), title='$K=4$, profiles from the lowest to the highest burden', title_fontsize=6.5, ncol=4,
               loc='lower center', bbox_to_anchor=(0.5, -0.1), frameon=False, handlelength=1.2, columnspacing=1.5)
    fig.tight_layout(w_pad=0.4)
    save(fig, 'fig_pfas_groups')

for name, (xlabel, labels, groups) in GROUPS.items():
    print(f'{name}: n = {[int(s.sum()) for s in groups]}, weighted share = {[round(float(q[s].sum()), 3) for s in groups]}')
    fig, ax = plt.subplots(1, len(KS), figsize=(4.2 * len(KS), 4.4), sharey=True)
    for A, K in zip(ax, KS):
        Ag = group_shares(K, groups)
        m, (lo, hi), col, wd = np.median(Ag, 0), pct(Ag), cols(K), .8 / K
        for j in range(K):
            xs = np.arange(len(groups)) + (j - (K - 1) / 2) * wd
            A.bar(xs, m[:, j], wd, color=col[j], label=f'profile {j + 1}')
            A.errorbar(xs, m[:, j], yerr=[m[:, j] - lo[:, j], hi[:, j] - m[:, j]], fmt='none', ecolor='k', lw=.7, capsize=1.5)
        A.set(xticks=range(len(groups)), xlabel=xlabel, title=f'K = {K}', ylim=(0, 1))
        A.set_xticklabels(labels, rotation=20 if name != 'pir' else 0, ha='right' if name != 'pir' else 'center')
        A.legend(fontsize=8, loc='upper left', title='lowest → highest', title_fontsize=8)
    ax[0].set_ylabel('share of the group in each profile')
    fig.suptitle(f'Allocation of {xlabel} to the representative profiles (95% intervals from the design-calibrated CoRP posterior draws)')
    fig.tight_layout()
    pass

age group (years) median shares [[0.18, 0.61, 0.19, 0.01], [0.15, 0.39, 0.37, 0.09], [0.11, 0.26, 0.44, 0.19], [0.03, 0.14, 0.37, 0.46]] lo [[0.04, 0.52, 0.11, 0.01], [0.07, 0.35, 0.31, 0.06], [0.06, 0.23, 0.4, 0.14], [0.01, 0.1, 0.35, 0.38]] hi [[0.37, 0.67, 0.31, 0.03], [0.25, 0.41, 0.44, 0.13], [0.17, 0.3, 0.47, 0.26], [0.05, 0.2, 0.39, 0.53]]
race and Hispanic origin median shares [[0.19, 0.38, 0.31, 0.13], [0.07, 0.28, 0.41, 0.24], [0.16, 0.38, 0.27, 0.18], [0.13, 0.27, 0.37, 0.21], [0.16, 0.37, 0.29, 0.18]] lo [[0.07, 0.35, 0.25, 0.1], [0.03, 0.24, 0.38, 0.18], [0.08, 0.32, 0.23, 0.16], [0.08, 0.24, 0.3, 0.2], [0.1, 0.32, 0.23, 0.14]] hi [[0.27, 0.42, 0.39, 0.16], [0.13, 0.31, 0.44, 0.31], [0.29, 0.41, 0.34, 0.22], [0.22, 0.3, 0.42, 0.27], [0.25, 0.4, 0.38, 0.2]]
education, adults aged 20+ median shares [[0.14, 0.26, 0.37, 0.23], [0.11, 0.29, 0.38, 0.22], [0.1, 0.27, 0.4, 0.22], [0.08, 0.25, 0.41, 0.27]] lo [[0.04, 0.24, 0.33, 0.19], [0.06, 0.26, 0.33, 0.19], [0.06, 0.25, 0.36, 0

In [14]:
SEL = ['age', 'race', 'pir']                                              # supplementary figure: the same shares for K = 3..7
with plt.rc_context(PAPER):
    fig, ax = plt.subplots(len(SEL), len(KS), figsize=(PW, 1.7 * len(SEL)), sharey='row')
    for r, name in enumerate(SEL):
        xlabel, labels, groups = GROUPS[name]
        for A, K in zip(ax[r], KS):
            Ag = group_shares(K, groups)
            m, (lo, hi), col, wd = np.median(Ag, 0), pct(Ag), cols(K), .8 / K
            for j in range(K):
                xs = np.arange(len(groups)) + (j - (K - 1) / 2) * wd
                A.bar(xs, m[:, j], wd, color=col[j])
                A.errorbar(xs, m[:, j], yerr=[m[:, j] - lo[:, j], hi[:, j] - m[:, j]], fmt='none', ecolor='k', lw=.4, capsize=.8)
            A.set(xticks=range(len(groups)), ylim=(0, 0.8))
            A.set_xticklabels(labels, rotation=30, ha='right', fontsize=5)
            if r == 0: A.set_title(f'$K={K}$')
            if K == KS[0]: A.set_ylabel('share of group')
            if r == len(SEL) - 1 and K == KS[2]: A.set_xlabel('')
            print(f'{name} K={K}: mean profile rank (1..K) by group', [round(float((m[g] * np.arange(1, K + 1)).sum()), 2) for g in range(len(groups))])
        ax[r, 2].set_xlabel(xlabel.replace(' group (years)', ' (years)').replace('family ', ''))
    fig.tight_layout(w_pad=0.3, h_pad=0.5)
    save(fig, 'fig_pfas_groups_byK')


age K=3: mean profile rank (1..K) by group [1.63, 1.88, 2.18, 2.57]
age K=4: mean profile rank (1..K) by group [2.03, 2.39, 2.7, 3.24]
age K=5: mean profile rank (1..K) by group [2.6, 3.06, 3.4, 4.08]
age K=6: mean profile rank (1..K) by group [2.83, 3.4, 3.91, 4.76]
age K=7: mean profile rank (1..K) by group [3.0, 3.71, 4.31, 5.35]
race K=3: mean profile rank (1..K) by group [1.91, 2.23, 1.95, 2.16, 1.99]
race K=4: mean profile rank (1..K) by group [2.4, 2.82, 2.46, 2.64, 2.47]
race K=5: mean profile rank (1..K) by group [3.06, 3.56, 3.1, 3.39, 3.12]
race K=6: mean profile rank (1..K) by group [3.41, 4.08, 3.48, 3.89, 3.6]
race K=7: mean profile rank (1..K) by group [3.72, 4.5, 3.86, 4.27, 3.87]
pir K=3: mean profile rank (1..K) by group [1.88, 2.14, 2.24]
pir K=4: mean profile rank (1..K) by group [2.37, 2.67, 2.86]
pir K=5: mean profile rank (1..K) by group [2.98, 3.4, 3.61]
pir K=6: mean profile rank (1..K) by group [3.35, 3.87, 4.13]
pir K=7: mean profile rank (1..K) by group [3.6

## Figure 8. Comparison with other representative-point methods

The Introduction reviews five kinds of representative points. Four can be computed from the survey-weighted sample $\widehat P_n=\sum_iq_i\delta_{x_i}$ on the centered log scale and are compared here with the WAC posterior means $(\bar z,\bar w)$.

* Principal points (Flury, 1990), estimated by weighted $k$-means: the $K$ points minimizing $\sum_iq_i\min_j\|x_i-z_j\|^2$, best of 50 Lloyd starts, with the Voronoi masses as weights. This is the optimal quantizer of $\widehat P_n$, the point estimate $\widehat\theta_n$ around which the WAC posterior concentrates (Theorem 4), so it is the frequentist counterpart of WAC rather than a competitor with a different target.
* Support points (Mak and Joseph, 2018): $K$ equal-weight points minimizing the energy distance to $\widehat P_n$, by the convex-concave iteration of the paper (their equation (22), with the survey weights in the sample terms), best of 5 starts. The authors' R package `support` (version 0.1.7) runs the same iteration. Run on these data with the same weights and 5 starts, it gives the same energy distance for $K=3,4$ and one at most 0.6% larger for $K=5,6,7$. By default the package first scales each coordinate to unit variance. Here the coordinates are left on the log scale, as for the other methods, and with the scaling $W_2^2$, the quantization error and the energy distance change by at most 2.1%.
* Kernel herding (Chen, Welling and Smola, 2010): $K$ observations chosen one at a time by their equation (8), each maximizing the kernel mean of $\widehat P_n$ minus the average kernel to the points already chosen, with a Gaussian kernel and equal weights. The search is over the observations, as in Section 3.2 of the paper. The paper does not prescribe a bandwidth, and $h$ is set here to the median pairwise distance.
* Archetypal analysis (Cutler and Breiman, 1994): $K$ convex combinations of the observations minimizing $\sum_iq_i\|x_i-\sum_j\alpha_{ij}a_j\|^2$, by the alternating algorithm of the paper (Section 4: the memberships given the archetypes, then the archetypes one at a time given the memberships, each a least squares problem on the simplex solved by penalized non-negative least squares), best of 20 starts. The weight of archetype $j$ is the mean membership $\sum_iq_i\alpha_{ij}$.

Two are not applicable. Stein points need the score $\nabla\log p_0$ of the target, which is unknown here. A coreset is a weighted subset that preserves a given loss; for the quantization loss it only approximates the problem that WAC solves on the full data.

All five representations are scored against $\widehat P_n$ by four discrepancies: $W_2^2$ with the method's own weights, the quantization error $\sum_iq_i\min_j\|x_i-z_j\|^2$ (the same atoms with their best weights, the Voronoi masses), the energy distance, and the squared MMD with bandwidth $h$. Each method is expected to do best on the criterion it optimizes.

In [15]:
import rep_points as rp
h, eq = rp.bandwidth(Y, q), lambda K: np.full(K, 1 / K)
order = lambda z, w: (z[np.argsort(z.sum(1))], w[np.argsort(z.sum(1))])   # lowest to highest overall burden
if RECOMPUTE:
    with get_context('fork').Pool(8) as pool:
        AR = {K: min(pool.starmap(rp.archetypes, [(Y, q, K, 200, s) for s in range(20)]), key=lambda r: r[2]) for K in KS}
    np.savez(os.path.join(RES, 'archetypes.npz'), **{f'K{K}_{k}': v for K, r in AR.items() for k, v in zip('Aw', (r[0], q @ r[1]))})
AR = np.load(os.path.join(RES, 'archetypes.npz'))
REP = {}
for K in KS:
    sp = min((rp.support_points(Y, q, K, seed=s) for s in range(5)), key=lambda z: rp.discrepancies(Y, q, z, eq(K), h)['energy'])
    Mq = Model(Y, 2, weights=q)
    km = quantizer(Mq, K, n_init=50)[1]
    REP[K] = {'CoRP': (C[f'K{K}_s0_zbar'], C[f'K{K}_s0_wbar']), 'k-means': order(km, Mq.voronoi_masses(km)), 'support points': order(sp, eq(K)),
              'kernel herding': order(rp.herding(Y, q, K, h), eq(K)), 'archetypes': order(AR[f'K{K}_A'], AR[f'K{K}_w'])}
DIS = {(K, m): rp.discrepancies(Y, q, z, w, h) for K in KS for m, (z, w) in REP[K].items()}
CRIT = {'W2': '$W_2^2$, own weights', 'quant': 'quantization error', 'energy': 'energy distance', 'mmd': f'MMD$^2$, $h$ = {h:.2f}'}
METH = list(REP[3])
print(f'{"K":>2s} {"method":15s} ' + ' '.join(f'{k:>8s}' for k in CRIT))
for K in KS:
    for m in METH:
        print(f'{K:2d} {m:15s} ' + ' '.join(f'{DIS[K, m][k]:8.4f}' for k in CRIT))

mc = dict(zip(METH, [LINE[2], LINE[4], LINE[1], LINE[5], LINE[0]]))
with plt.rc_context(PAPER):
    fig = plt.figure(figsize=(PW, 5.0))
    g1 = fig.add_gridspec(1, 4, left=0.08, right=0.99, top=0.88, bottom=0.62, wspace=0.45)
    g2 = fig.add_gridspec(1, 5, left=0.08, right=0.99, top=0.5, bottom=0.14, wspace=0.12)
    ax = {**{k: fig.add_subplot(g1[0, i]) for i, k in enumerate(CRIT)}, **{f'p{m}': fig.add_subplot(g2[0, i]) for i, m in enumerate(METH)}}
    for k, lab in CRIT.items():
        for m in METH:
            ax[k].plot(list(KS), [DIS[K, m][k] for K in KS], 's--' if m == 'k-means' else 'o-', color=mc[m], label=m, ms=5 if m == 'k-means' else 3,
                       mfc='none' if m == 'k-means' else mc[m])
        ax[k].set(xticks=list(KS), xlabel='$K$', yscale='log', title=lab)
    fig.legend(*ax['W2'].get_legend_handles_labels(), loc='upper center', ncol=5, frameon=False, bbox_to_anchor=(0.5, 1.0))
    for m in METH:
        A, (z, w) = ax[f'p{m}'], REP[4][m]
        for j in range(4):
            A.plot(x5, np.exp(z[j] + mu), color=cols(4)[j], lw=0.6 + 7 * w[j], marker='o', ms=1.5 + 7 * w[j], label=f'{w[j]:.2f}')
        A.set(yscale='log', xticks=x5, title=m)
        A.set_xticklabels(NAMES, rotation=50, ha='right', fontsize=5.5)
        A.legend(title='$w_k$', title_fontsize=5.5, fontsize=5, loc='upper left', handlelength=1.2, borderpad=0.2, labelspacing=0.15)
        if m != 'archetypes':
            A.sharey(ax[f'p{METH[0]}'])
            A.tick_params(labelleft=m == METH[0])
    ax[f'p{METH[0]}'].set_ylabel('ng/mL, $K=4$')
    save(fig, 'fig_pfas_methods')

 K method                W2    quant   energy      mmd
 3 CoRP              1.3315   1.3314   0.2101   0.0089
 3 k-means           1.3314   1.3314   0.2117   0.0090
 3 support points    1.3811   1.3556   0.1992   0.0095
 3 kernel herding    1.5054   1.4773   0.2228   0.0138
 3 archetypes        9.1257   7.7013   1.6095   0.5062
 4 CoRP              1.1032   1.1032   0.1605   0.0073
 4 k-means           1.1031   1.1031   0.1606   0.0073
 4 support points    1.2162   1.2002   0.1414   0.0061
 4 kernel herding    1.4410   1.4251   0.2101   0.0191
 4 archetypes        8.6565   6.6105   1.3218   0.4150
 5 CoRP              0.9845   0.9845   0.1324   0.0069
 5 k-means           0.9844   0.9844   0.1318   0.0069
 5 support points    1.0940   1.0758   0.1079   0.0045
 5 kernel herding    1.5663   1.3915   0.2249   0.0236
 5 archetypes       10.1903   9.8055   1.5061   0.5098
 6 CoRP              0.9164   0.9163   0.1038   0.0057
 6 k-means           0.9156   0.9156   0.1040   0.0056
 6 support

CoRP and weighted $k$-means agree: their $W_2^2$ and quantization error differ by at most 0.15%, their energy distance by at most 0.8% and their MMD by at most 3%, the locations by at most 0.045 log units (less than one posterior standard deviation) and the weights by at most 0.015. This is expected, since the CoRP posterior concentrates around the optimal quantizer of $\widehat P_n$. What CoRP adds to $k$-means is the posterior of $K$ and calibrated uncertainty for the locations and weights.

Among the methods with other targets, CoRP and support points each do best on the criterion they optimize. For every $K$, CoRP has $W_2^2$ 4% to 11% and quantization error 2% to 10% below support points, and support points have the smallest energy distance for every $K$ (5% to 19% below CoRP) and the smallest MMD for $K\ge4$. At $K=4$ both give profiles ordered mainly by overall burden (the two middle support points cross), and they differ in the weights: those of support points are equal by construction, whereas CoRP estimates them (0.11 to 0.37). Kernel herding is worse than both on all four criteria. Its weights are equal and its points are chosen one at a time, which Chen, Welling and Smola note is not optimal when the number of points is fixed in advance. Archetypes are extreme profiles at the edge of the data, for example, at $K=4$, one with all five concentrations at or near the fill value for non-detects and one with n-PFOS at 86 ng/mL, so their $W_2^2$, quantization error and energy distance are 6 to 14 times those of CoRP and their MMD 57 to 82 times. They describe the extremes of exposure rather than typical profiles. None of the four alternatives provides a posterior of $K$ or uncertainty for the points and weights, which would require a separate bootstrap.

## Figure 9. Sensitivity to non-detects

164 of the 2551 participants (6.4%, weighted 6.9%) have at least one concentration below the limit of detection (LOD, 0.1 ng/mL for all five PFAS), mostly PFNA (5.4%). NHANES reports each non-detect as $\mathrm{LOD}/\sqrt2\approx0.07$, and the main analysis uses these values as if they were measured. Two alternatives are compared with it, each refitted with the same CoRP model (fixed-$K$ chain of 3000 iterations, design sandwich, $t_{15}$ widening) for $K=3,\dots,7$:

* **Censored.** A non-detect only says that the concentration lies below the LOD. Log concentrations are modeled as weighted multivariate normal, with each non-detect left-censored at $\log\mathrm{LOD}$, and fitted by data augmentation on all participants: every non-detect is redrawn from its normal distribution conditional on the participant's other four values, truncated above at $\log\mathrm{LOD}$, and the mean and covariance are then refitted. After 50 burn-in sweeps, 10 completed data sets are kept, 15 sweeps apart. CoRP is run on each, and the posterior draws of the 10 runs are pooled, so the spread includes the uncertainty about the censored values.
* **Complete case.** The 164 participants with any non-detect are dropped, which is not recommended and is shown only for comparison. They are not missing at random: by definition they have the lowest exposures, so dropping them removes mass from the bottom of the distribution.

In [16]:
ND_FILE, NM, nd, cc = os.path.join(RES, 'nondetect.npz'), 10, F == 1, ~(F == 1).any(1)


def censored_imputations(M=NM, burn=50, gap=15, seed=0):
    """Data augmentation for a weighted multivariate normal on log concentrations with every non-detect left-censored
    at log LOD; returns M completed data sets (centred like Y), gap sweeps apart after burn-in."""
    rng, L, out = np.random.default_rng(seed), np.log(X), []
    for t in range(burn + gap * M):
        m, S = q @ L, np.cov(L.T, aweights=q)
        for j in range(5):
            o = np.arange(5) != j
            beta = np.linalg.solve(S[np.ix_(o, o)], S[o, j])
            sd = np.sqrt(S[j, j] - S[j, o] @ beta)
            mj = m[j] + (L[nd[:, j]][:, o] - m[o]) @ beta
            L[nd[:, j], j] = truncnorm.rvs(-np.inf, (np.log(LOD) - mj) / sd, loc=mj, scale=sd, random_state=rng)
        if t >= burn and (t - burn) % gap == gap - 1:
            out.append(L - mu)
    return out


def nd_job(name, x, rows, K):
    """CoRP fixed-K chain on data x (rows: participants kept), calibrated with the design sandwich and labeled like the
    main seed-0 chain; K = 0 instead returns the weighted distortion L_K, K = 1..9."""
    wt = q[rows]
    if K == 0:
        return name, 0, [quantizer(Model(x, 2, weights=wt), k)[0] for k in range(1, 10)]
    M = Model(x, 2, Exp(len(x) * LAM0), penalty(0.0), GaussianAtoms(np.zeros(5), 2 * Y.std(0).mean()), Dirichlet(1.0),
              K_max=16, weights=wt)
    r = run(M, 3000, lloyd_start(M, K, 0), split_merge=False, temper=False, burn=1 / 3, thin=2, seed=0)
    cd = calibrate(M, r['draws'], K, design=(stra[rows], psu[rows]))
    p = relabel(C[f'K{K}_s0_zbar'], cd['z_bar'])
    return name, K, (cd['z_bar'][p], cd['w_bar'][p], cd['z'][:, p], cd['w'][:, p])


if RECOMPUTE or not os.path.exists(ND_FILE):
    imp, allr = censored_imputations(), np.ones(n, bool)
    jobs = [(f'cens{m}', x, allr, K) for m, x in enumerate(imp) for K in (0, *KS)] + [('cc', Y[cc], cc, K) for K in (0, *KS)]
    with get_context('fork').Pool(8) as pool:
        res = {(nm, K): v for nm, K, v in pool.starmap(nd_job, jobs)}
    out = {'L_cc': res['cc', 0], 'L_cens': np.mean([res[f'cens{m}', 0] for m in range(NM)], 0),
           'imp_mean': np.mean([x[nd] for x in imp], 0)}
    for K in KS:
        R = [res[f'cens{m}', K] for m in range(NM)]
        out |= {f'K{K}_cens_zbar': np.mean([r[0] for r in R], 0), f'K{K}_cens_wbar': np.mean([r[1] for r in R], 0),
                f'K{K}_cens_z': np.concatenate([r[2] for r in R]), f'K{K}_cens_w': np.concatenate([r[3] for r in R])}
        out |= dict(zip([f'K{K}_cc_zbar', f'K{K}_cc_wbar', f'K{K}_cc_z', f'K{K}_cc_w'], res['cc', K]))
    np.savez(ND_FILE, **out)
ND = np.load(ND_FILE)


def get_nd(K, v):
    """posterior means (log ng/mL) and t_15-widened draws of z_k, w_k under treatment v ('cens' or 'cc')."""
    zb, wb = ND[f'K{K}_{v}_zbar'], ND[f'K{K}_{v}_wbar']
    return zb + mu, wb, zb + mu + TF * (ND[f'K{K}_{v}_z'] - zb), wb + TF * (ND[f'K{K}_{v}_w'] - wb)


print(f'censored non-detects: mean imputed value {np.exp(ND["imp_mean"] + mu[np.where(nd)[1]]).mean():.3f} ng/mL '
      f'(NHANES fill value {LOD / np.sqrt(2):.3f}); complete case keeps {cc.sum()} of {n} participants')
print('resolution c_K = (L_{K-1} - L_{K+1}) / 2 for K* = 3..7:')
for lab, L in (('main', Ls[:9]), ('censored', ND['L_cens']), ('complete case', ND['L_cc'])):
    print(f'  {lab:14s}', {K: round(float(L[K - 2] - L[K]) / 2, 3) for K in KS})
print('shift from the main analysis: max |z_k| change in log units / in posterior sd, and w_k change')
for K in KS:
    zb, wb, Z, W = get(K)
    for v, lab in (('cens', 'censored'), ('cc', 'complete case')):
        z2, w2, _, _ = get_nd(K, v)
        d = np.abs(z2 - zb)
        print(f'  K {K} {lab:14s} z: {d.max(1).round(2).tolist()} / {(d / Z.std(0)).max(1).round(1).tolist()};  w: {(w2 - wb).round(3).tolist()}')

TRT = {'main (LOD/√2 as measured)': None, 'censored (pooled over 10 imputations)': 'cens', 'complete case (drop non-detects)': 'cc'}
tc = dict(zip(TRT, ['0.25', '#2a6fdb', '#d9822b']))
K = 4
fig, ax = plt.subplots(1, K + 1, figsize=(4.1 * (K + 1), 4.6), gridspec_kw=dict(width_ratios=[1] * K + [1.1]))
for t, v in TRT.items():
    zb, wb, Z, W = get(K) if v is None else get_nd(K, v)
    for j in range(K):
        ax[j].fill_between(x5, *np.exp(pct(Z[:, j])), color=tc[t], alpha=.18, lw=0)
        ax[j].plot(x5, np.exp(zb[j]), 'o-', color=tc[t], lw=1.8, ms=4, label=t)
    lo, hi = pct(W)
    off = list(TRT).index(t) * .2 - .2
    ax[K].errorbar(wb, np.arange(K) + off, xerr=[wb - lo, hi - wb], fmt='o', color=tc[t], capsize=3, label=t)
for j in range(K):
    ax[j].set(yscale='log', xticks=x5, title=f'profile {j + 1}' + (' (lowest)' if j == 0 else ' (highest)' if j == K - 1 else ''))
    ax[j].set_xticklabels(NAMES, rotation=40, ha='right')
ax[0].set_ylabel('serum concentration (ng/mL, log scale)')
ax[K].set(yticks=range(K), yticklabels=[f'profile {j + 1}' for j in range(K)], xlabel='population share $w_k$', title='weights, 95% intervals')
ax[K].legend(loc='upper center', bbox_to_anchor=(.5, -.18), fontsize=8)
fig.suptitle(f'Sensitivity of the K = {K} profiles to the treatment of non-detects (lines: posterior means; bands: pointwise 95% intervals)')
fig.tight_layout()

censored non-detects: mean imputed value 0.063 ng/mL (NHANES fill value 0.071); complete case keeps 2387 of 2551 participants
resolution c_K = (L_{K-1} - L_{K+1}) / 2 for K* = 3..7:
  main           {3: 0.338, 4: 0.174, 5: 0.094, 6: 0.065, 7: 0.052}
  censored       {3: 0.36, 4: 0.192, 5: 0.103, 6: 0.067, 7: 0.053}
  complete case  {3: 0.238, 4: 0.103, 5: 0.067, 6: 0.054, 7: 0.045}
shift from the main analysis: max |z_k| change in log units / in posterior sd, and w_k change
  K 3 censored       z: [0.14, 0.08, 0.04] / [0.9, 0.6, 0.4];  w: [-0.034, 0.008, 0.026]
  K 3 complete case  z: [0.48, 0.36, 0.22] / [3.1, 2.5, 2.5];  w: [0.118, -0.009, -0.109]
  K 4 censored       z: [0.53, 0.14, 0.06, 0.03] / [4.3, 1.1, 0.8, 0.5];  w: [-0.051, 0.013, 0.023, 0.015]
  K 4 complete case  z: [0.64, 0.35, 0.17, 0.08] / [6.5, 2.9, 2.0, 1.2];  w: [0.092, -0.024, -0.047, -0.021]
  K 5 censored       z: [0.42, 0.08, 0.06, 0.03, 0.01] / [8.9, 1.4, 0.9, 0.5, 0.2];  w: [-0.011, -0.012, 0.008, 0.011, 0.005]


Treating non-detects as censored changes only the lowest profile. The censored model imputes non-detects at 0.063 ng/mL on average, slightly below the fill value 0.071, so the lowest profile moves down (by up to 0.5 log units at $K=4$, mostly in the coordinates with most non-detects) and its weight falls (0.107 to 0.056 at $K=4$). Profiles 2 to 4 move by less than 0.15 log units, and for $K\ge5$ every profile except the lowest stays within 2.5 posterior standard deviations. The resolution scale barely changes ($c_4$ 0.174 to 0.192).

Dropping the participants with non-detects changes much more. Removing the lowest exposures shifts the whole bottom of the distribution: the two lowest profiles move up by 0.35 to 1.1 log units (2.5 to 29 posterior standard deviations), the weights are redistributed (at $K=4$ the lowest profile grows from 0.107 to 0.199, while profiles 2 to 4 lose mass), and the distortion curve flattens, so every $c_K$ falls (for example $c_4$ from 0.174 to 0.103) and a given price $c$ selects more representatives. The complete-case analysis therefore describes a different, higher-exposed population, and the censored analysis is the appropriate sensitivity check. Its agreement with the main analysis for all but the lowest profile shows that the fill value $\mathrm{LOD}/\sqrt2$ matters only for describing the least exposed participants.